[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/v08_planes.ipynb)

# Planes and the triple product: rock the table

A café table with four legs often rocks on an old tiled floor, while a stool with three legs never does. Three points
always lie on one plane; a fourth has to be lucky, and the scalar triple product tells you exactly how unlucky it was.

In this notebook you can put the table on its model floor, raise the fourth tile until the box made by three arrows
goes flat, and tilt a plane by its normal while its equation changes.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from ipywidgets import interact, FloatSlider, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # along (dot) · the result (the box) · across (cross, the normal)

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def dot(a, b): return sum(x * y for x, y in zip(a, b))
def cross(a, b): return (a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0])
def length(a): return math.sqrt(dot(a, a))
def sub(a, b): return tuple(x - y for x, y in zip(a, b))
def add(a, b): return tuple(x + y for x, y in zip(a, b))
def triple(a, b, c): return dot(a, cross(b, c))

def nice(v):
    """A number as the board shows it: 3, −1, 0.25."""
    s = f"{v:.2f}".rstrip("0").rstrip(".") if abs(v - round(v)) > 1e-9 else str(int(round(v)))
    return "0" if s in ("-0", "−0") else s.replace("-", "−")

def column(fig, x, y, vals, color=CHALK, size=15, label=""):
    """An arrow's column of numbers, in a bracket box, at figure position (x, y)."""
    fig.text(x, y, "\n".join(nice(v) for v in vals), color=color, fontsize=size, va="center", ha="center",
             family="serif", linespacing=1.45, bbox=dict(boxstyle="square,pad=0.4", fc=SLATE, ec=DIM, lw=1.2))
    if label: fig.text(x, y + 0.13, label, color=DIM, fontsize=12, ha="center", family="serif")

def space(ax, elev, azim, unit=" (cm)"):
    """A 3D stage on the slate: dim panes, chalk labels, the view the sliders choose."""
    ax.set_facecolor(SLATE)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0.16, 0.22, 0.2, 0.6)); axis.label.set_color(DIM)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("x" + unit); ax.set_ylabel("y" + unit); ax.set_zlabel("z" + unit)

## 1 · The table on its model floor

A model, in centimetres. The four feet stand on tiles at (0, 0, 0), (60, 0, 1), (0, 60, 1) and (60, 60, 0). The
first three feet fix a plane (shaded); its normal, the arrow square to it, is the cross product of the two arrows from
the first foot: (60, 0, 1) × (0, 60, 1) = (−60, −60, 3600), or (−1, −1, 60) after dividing by sixty. The table rests
on those three feet, so the fourth foot (coral) hangs above its tile. Heights are stretched so you can see them; the
legs are drawn short.

In [ ]:
P1, P2, P3 = (0, 0, 0), (60, 0, 1), (0, 60, 1)
U, V = sub(P2, P1), sub(P3, P1)
N = cross(U, V)                                   # (−60, −60, 3600)
n = tuple(x / 60 for x in N)                      # (−1, −1, 60)

def plane_z(x, y):
    """Height of the plane through the three good feet: −x − y + 60z = 0."""
    return (x + y) / 60

def table(ax, h4, lim=75):
    """The floor, its four tiles, the plane of the three good feet, the table on them, and the fourth foot."""
    feet = [P1, P2, P3, (60, 60, h4)]
    for (fx, fy, fz), col in zip(feet, [DIM, DIM, DIM, CORAL]):           # the tiles, 30 cm square
        xs, ys = np.meshgrid([fx - 15, fx + 15], [fy - 15, fy + 15])
        ax.plot_surface(xs, ys, np.full_like(xs, fz, dtype=float), color=col, alpha=0.35, shade=False)
    gx, gy = np.meshgrid(np.linspace(-15, lim, 2), np.linspace(-15, lim, 2))
    ax.plot_surface(gx, gy, plane_z(gx, gy), color=TEAL, alpha=0.18, shade=False)
    tips = [(x, y, plane_z(x, y)) for x, y, _ in feet]                   # a rigid table: its leg tips share one plane
    tops = [(x, y, z + 2.5) for x, y, z in tips]
    for (x, y, z), (_, _, zt) in zip(tips, tops):
        ax.plot([x, x], [y, y], [z, zt], color=CHALK, lw=3)
    ring = tops[:2] + [tops[3], tops[2], tops[0]]
    ax.plot(*zip(*ring), color=CHALK, lw=3)
    for x, y, z in feet[:3]: ax.scatter(x, y, z, color=CHALK, s=40, depthshade=False)
    gap = plane_z(60, 60) - h4
    ax.scatter(60, 60, h4, color=CORAL, s=50, depthshade=False)
    if gap > 0.01: ax.plot([60, 60], [60, 60], [h4, plane_z(60, 60)], color=CORAL, lw=2, ls="--")
    c = (20, 20, plane_z(20, 20))
    ax.quiver(*c, *(k * 0.06 for k in n), color=LILAC, lw=2.4, arrow_length_ratio=0.25)
    ax.set_xlim(-15, lim); ax.set_ylim(-15, lim); ax.set_zlim(-1.5, 6)
    return gap

def floor_view(turn=-60, tilt=22):
    fig = plt.figure(figsize=(9, 5.6))
    ax = fig.add_axes([0.0, 0.0, 0.68, 1.0], projection="3d")
    space(ax, tilt, turn)
    gap = table(ax, 0.0)
    ax.set_title(f"the fourth foot hangs {nice(round(gap, 2))} cm above its tile", color=CORAL)
    column(fig, 0.73, 0.55, U, label="u")
    column(fig, 0.82, 0.55, V, label="v")
    column(fig, 0.91, 0.55, n, color=LILAC, label="normal")
    plt.show()

In [ ]:
interact(floor_view, turn=FloatSlider(value=-60, min=-180, max=180, step=10, description="view: turn"),
         tilt=FloatSlider(value=22, min=-10, max=80, step=4, description="view: tilt"));

## 2 · Raise the fourth tile: the box swells and flattens

The three arrows from the first foot to the others, u = (60, 0, 1), v = (0, 60, 1) and w = (60, 60, h), make a
slanted box (green), reaching out to u + v + w. Its volume is the scalar triple product (u × v) · w, the floor of the box times how high its lid
stands. Drag the fourth tile's height h and watch the box. At one height the volume reads zero: the box is flat, the
three arrows lie in one plane, and the fourth foot touches.

In [ ]:
def box(ax, a, b, c, color=SAGE):
    """The slanted box on three arrows from the first foot."""
    o = np.array(P1, float); a, b, c = (np.array(v, float) for v in (a, b, c))
    faces = [[o, o + a, o + a + b, o + b], [o + c, o + a + c, o + a + b + c, o + b + c],
             [o, o + a, o + a + c, o + c], [o + b, o + a + b, o + a + b + c, o + b + c],
             [o, o + b, o + b + c, o + c], [o + a, o + a + b, o + a + b + c, o + a + c]]
    ax.add_collection3d(Poly3DCollection(faces, facecolor=color, edgecolor=color, alpha=0.22, linewidths=1.2))

def rock(fourth_tile_height=0.0, turn=-60, tilt=22):
    h = fourth_tile_height
    W = (60, 60, h)
    vol = triple(U, V, W)                          # (u × v) · w = −7200 + 3600 h
    fig = plt.figure(figsize=(9, 5.6))
    ax = fig.add_axes([0.0, 0.0, 0.68, 1.0], projection="3d")
    space(ax, tilt, turn)
    gap = table(ax, h, 125)
    box(ax, U, V, W)
    if abs(vol) < 1e-9:
        ax.set_title("box volume 0 cm³: flat, and the fourth foot touches", color=CORAL)
    elif gap > 0:
        ax.set_title(f"box volume {nice(abs(vol))} cm³: the fourth foot hangs {nice(gap)} cm up", color=CHALK)
    else:
        ax.set_title(f"box volume {nice(abs(vol))} cm³: this tile is too high, another foot lifts", color=CHALK)
    column(fig, 0.73, 0.55, U, label="u")
    column(fig, 0.82, 0.55, V, label="v")
    column(fig, 0.91, 0.55, W, color=CORAL, label="w")
    plt.show()
    print(f"(u × v) · w = (−60)(60) + (−60)(60) + (3600)({nice(h)}) = {nice(vol)}")

In [ ]:
interact(rock, fourth_tile_height=FloatSlider(value=0.0, min=-1.0, max=3.0, step=0.5, description="4th tile h",
                                              layout=Layout(width="520px")),
         turn=FloatSlider(value=-60, min=-180, max=180, step=10, description="view: turn"),
         tilt=FloatSlider(value=22, min=-10, max=80, step=4, description="view: tilt"));

## 3 · Tilt a plane by its normal

A plane is fixed by one point on it and its normal n = (a, b, c), the way it faces. Every point r on it has the same
share along the normal, r · n = d, which in x, y and z reads ax + by + cz = d. Here the plane goes through the point
A(1, 0, 1). Drag the normal's numbers and watch the plane turn and its equation change. It starts as the plane of the
exam question in the video, x + 2y + 2z = 3.

In [ ]:
A = (1, 0, 1)

def tilt_plane(a=1.0, b=2.0, c=2.0, turn=-55, tilt=20):
    nv = (a, b, c)
    if length(nv) < 1e-9:
        print("The normal can't be the zero arrow: give it some length."); return
    d = dot(nv, A)
    fig = plt.figure(figsize=(9, 5.6))
    ax = fig.add_axes([0.0, 0.0, 0.68, 1.0], projection="3d")
    space(ax, tilt, turn, "")
    k = int(np.argmax([abs(a), abs(b), abs(c)]))   # solve for the coordinate the normal leans on most
    s = np.linspace(-3, 3, 13)
    p, q = np.meshgrid(s, s)
    if k == 2: X, Y, Z = p, q, (d - a * p - b * q) / c
    elif k == 1: X, Z, Y = p, q, (d - a * p - c * q) / b
    else: Y, Z, X = p, q, (d - b * p - c * q) / a
    keep = (np.abs(X) <= 3.01) & (np.abs(Y) <= 3.01) & (np.abs(Z) <= 3.01)
    ax.plot_surface(np.where(keep, X, np.nan), np.where(keep, Y, np.nan), np.where(keep, Z, np.nan),
                    color=TEAL, alpha=0.3, shade=False)
    u = np.array(nv) / length(nv) * 2
    ax.quiver(*A, *u, color=LILAC, lw=3, arrow_length_ratio=0.2)
    ax.scatter(*A, color=CORAL, s=60, depthshade=False)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_zlim(-3, 3)
    terms = [(a, "x"), (b, "y"), (c, "z")]
    eq = ""
    for coef, var in terms:
        if abs(coef) < 1e-9: continue
        sign = "−" if coef < 0 else "+"
        mag = "" if abs(abs(coef) - 1) < 1e-9 else nice(abs(coef))
        eq += (f"{'−' if coef < 0 else ''}{mag}{var}" if not eq else f" {sign} {mag}{var}")
    ax.set_title(f"{eq} = {nice(d)}", color=CORAL, fontsize=17)
    column(fig, 0.80, 0.55, nv, color=LILAC, label="normal n")
    fig.text(0.80, 0.25, f"d = n · A = {nice(d)}", color=CHALK, ha="center", fontsize=13, family="serif")
    plt.show()

In [ ]:
interact(tilt_plane, a=FloatSlider(value=1, min=-3, max=3, step=0.5, description="n: a"),
         b=FloatSlider(value=2, min=-3, max=3, step=0.5, description="n: b"),
         c=FloatSlider(value=2, min=-3, max=3, step=0.5, description="n: c"),
         turn=FloatSlider(value=-55, min=-180, max=180, step=10, description="view: turn"),
         tilt=FloatSlider(value=20, min=-10, max=80, step=4, description="view: tilt"));

## 4 · Do four points stand without a wobble?

Four points lie on one plane exactly when the box made by the three arrows from one of them to the others is flat:
the triple product is zero. Change the points and run the cell. It starts with the exam question from the video:
A(1, 0, 1), B(3, 0, 0), C(1, 1, 0) and D(2, 3, 4), whose box has volume 13.

In [ ]:
def flat(A, B, C, D):
    ab, ac, ad = sub(B, A), sub(C, A), sub(D, A)
    v = triple(ad, ab, ac)                         # (AB × AC) · AD
    print(f"AB = {ab}, AC = {ac}, AD = {ad}")
    print(f"(AB × AC) · AD = {nice(v)}:", "flat, all four on one plane" if abs(v) < 1e-9 else "a real box, so one point wobbles")

flat((1, 0, 1), (3, 0, 0), (1, 1, 0), (2, 3, 4))

**Try it.** The challenge from the video: keep A, B and C, and move the fourth point to D(0, 0, k). Which k lets all
four stand without a wobble? Work it out on paper (make the box flat), then check it with `flat` above.

## Sources
Euclid, *Elements* XI.2 (D. E. Joyce's online edition); J. C. Maxwell, "General considerations concerning scientific
apparatus", *Handbook to the Special Loan Collection of Scientific Apparatus* (South Kensington Museum, 1876); Tamil
Nadu Mathematics Class XII vol. 1, ch. 6 (Applications of Vector Algebra). All retrieved 2026-10-26. The floor is our
own model.